# Task 2 — Notebook 05: Areal statistics + run bundle

**Footprint vs native CDL:** Areas below use **one analysis-grid pixel** = GeoTIFF cell (side length ≈ √(`pixel_area_m2`); often **~320 m** for this stack). That yields **~10 ha/pixel** — a **regional cropland footprint on the analysis grid**, not a USDA acreage estimate from native **30 m** CDL.

**Per-state Corn Belt:** The **by-region** table assigns each classified pixel centroid to a state polygon from ``configs/task2_crop_rotation.yaml`` ``study_area.states`` (13-state Corn Belt), using the same boundary source as Notebook 04 (local shapefile if present, else Natural Earth). Pixels outside those state footprints are labeled ``outside_configured_states``. There is **no** Iowa/Nebraska longitude proxy.

Per-class **ha** = valid pixel count × `pixel_area_ha`. A companion **`areal_stats_metadata.json`** next to the CSV records CRS, transform-derived `pixel_area_ha`, and this disclaimer for reviewers.

**Output paths:** Areal **CSV + JSON** go to **`artifacts/tables/task4/`** (`configs/task2_crop_rotation.yaml` → `output.task4_tables_dir`). The per-state **bar chart** PNG still uses `output.figures_dir` (`artifacts/figures/task2/`). Rasters (GeoTIFFs) are **not** saved under `artifacts/tables/` — they remain under `data/processed/task2/`.

After the per-state CSV is written, the notebook saves **`task2__per_state_rotation_classes.png`** (stacked horizontal bars: regular / monoculture / irregular by state).


In [1]:
import json
import sys
import uuid
from datetime import date, datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import yaml
from rasterio.transform import Affine, xy as rio_xy

_cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "requirements.txt").is_file() and (p / "src").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not find repo root.")
sys.path.insert(0, str(REPO_ROOT))

from src.io.cdl_parquet import load_cdl_spatial_metadata
from src.viz.rotation_maps import load_cornbelt_state_boundaries_5070

with open(REPO_ROOT / "configs" / "task2_crop_rotation.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

sm_tif = REPO_ROOT / cfg["output"]["processed_dir"] / "rotation_class_map_smoothed.tif"
if not sm_tif.is_file():
    sm_tif = REPO_ROOT / cfg["output"]["processed_dir"] / "rotation_class_map.tif"

with rasterio.open(sm_tif) as src:
    flat = src.read(1).ravel()
    tr = src.transform
    crs_wkt = src.crs.to_wkt() if src.crs else None
    px_w, px_h = tr.a, tr.e
    pixel_area_m2 = abs(px_w * px_h)

pixel_area_ha = pixel_area_m2 / 10000.0
approx_res_m = float(np.sqrt(pixel_area_m2))
names = {0: "regular_rotation", 1: "monoculture", 2: "irregular", 255: "nodata"}
valid = flat[flat != 255]
rows = []
for cls in (0, 1, 2):
    cnt = int(np.sum(valid == cls))
    ha = cnt * pixel_area_ha
    pct = 100.0 * cnt / len(valid) if len(valid) else 0.0
    rows.append(
        {"rotation_class": names[cls], "pixel_count": cnt, "area_ha": round(ha, 1), "pct_of_valid": round(pct, 2)}
    )

overall_df = pd.DataFrame(rows)
try:
    print(overall_df.to_markdown(index=False))
except Exception:
    print(overall_df)

tbl_dir = REPO_ROOT / Path(cfg["output"].get("task4_tables_dir", cfg["output"]["tables_dir"]))
tbl_dir.mkdir(parents=True, exist_ok=True)
date_s = date.today().strftime("%Y%m%d")
csv_path = tbl_dir / f"task2__areal_stats_by_class__{date_s}.csv"
overall_df.to_csv(csv_path, index=False)
meta_path = tbl_dir / f"task2__areal_stats_by_class__{date_s}__metadata.json"
meta_doc = {
    "pixel_area_ha": round(pixel_area_ha, 4),
    "pixel_area_m2": round(pixel_area_m2, 2),
    "approx_grid_resolution_m": round(approx_res_m, 1),
    "crs_wkt": crs_wkt,
    "source_geotiff": str(sm_tif.relative_to(REPO_ROOT)).replace("\\", "/"),
    "note": (
        "Areas are total classified cropland footprint on the analysis grid (coarse cells), "
        "aggregated from native 30 m USDA CDL — not a direct USDA NASS acreage comparison."
    ),
    "companion_csv": str(csv_path.relative_to(REPO_ROOT)).replace("\\", "/"),
}
meta_path.write_text(json.dumps(meta_doc, indent=2), encoding="utf-8")
print("Wrote", csv_path.relative_to(REPO_ROOT))
print("Wrote", meta_path.relative_to(REPO_ROOT))

cls_pq = REPO_ROOT / cfg["output"]["processed_dir"] / "rotation_metrics_classified.parquet"
meta_sp = load_cdl_spatial_metadata(REPO_ROOT)
tlist = meta_sp["transform"]
aff = Affine(tlist[0], tlist[1], tlist[2], tlist[3], tlist[4], tlist[5])
crs_s = meta_sp.get("crs") or "EPSG:5070"
mdf = pd.read_parquet(cls_pq)
xs, ys = rio_xy(aff, mdf["iy"].to_numpy(), mdf["ix"].to_numpy(), offset="center")

reg_csv = None
region_rows = []
joined = None
try:
    import geopandas as gpd

    boundaries = load_cornbelt_state_boundaries_5070(REPO_ROOT)
    if boundaries is not None and not boundaries.empty:
        boundaries = boundaries.to_crs(crs_s)
        name_col = next((c for c in ("NAME", "name", "NAME_1") if c in boundaries.columns), None)
        if name_col is None:
            print("Corn Belt boundaries: no NAME/name column; using single-region fallback.")
        else:
            bnd = boundaries[[name_col, "geometry"]].copy()
            bnd["region"] = bnd[name_col].astype(str)
            bnd = bnd[["region", "geometry"]]
            pts = gpd.GeoDataFrame(mdf, geometry=gpd.points_from_xy(xs, ys), crs=crs_s)
            joined = gpd.sjoin(pts, bnd, how="left", predicate="within")
            if joined.index.duplicated().any():
                joined = joined[~joined.index.duplicated(keep="first")]
            joined["region"] = joined["region"].fillna("outside_configured_states")
            print("Per-state regions: Corn Belt polygons (task2 YAML / same source as Notebook 04).")
    else:
        print("Corn Belt boundaries unavailable (geopandas + Natural Earth or data/external/states/*.shp).")
except Exception as exc:
    print("Per-state Corn Belt split failed:", exc)
    joined = None

if joined is None or "region" not in getattr(joined, "columns", []):
    joined = mdf.copy()
    joined["region"] = "full_raster_extent"
    print("Single-region fallback: full_raster_extent (no per-state breakdown).")

for reg, g in joined.groupby("region"):
    vc = g["rotation_class"].value_counts(normalize=True)
    n = len(g)
    region_rows.append({
        "region": reg,
        "n_pixels": n,
        "pct_regular": round(100 * float(vc.get(0, 0)), 2),
        "pct_monoculture": round(100 * float(vc.get(1, 0)), 2),
        "pct_irregular": round(100 * float(vc.get(2, 0)), 2),
    })
reg_df = pd.DataFrame(region_rows).sort_values("region")
try:
    print(reg_df.to_markdown(index=False))
except Exception:
    print(reg_df)
reg_csv = tbl_dir / f"task2__areal_stats_by_region__{date_s}.csv"
reg_df.to_csv(reg_csv, index=False)
print("Wrote", reg_csv.relative_to(REPO_ROOT))

_plot_df = reg_df[~reg_df["region"].isin({"outside_configured_states", "full_raster_extent"})].copy()
if len(_plot_df) > 0:
    _plot_df = _plot_df.sort_values("pct_regular", ascending=True)
    _y = np.arange(len(_plot_df))
    _fig, _ax = plt.subplots(figsize=(10, max(6.0, 0.35 * len(_plot_df))))
    _pr = _plot_df["pct_regular"].to_numpy()
    _pm = _plot_df["pct_monoculture"].to_numpy()
    _pi = _plot_df["pct_irregular"].to_numpy()
    _ax.barh(_y, _pr, label="regular", color="#2ecc71")
    _ax.barh(_y, _pm, left=_pr, label="monoculture", color="#e74c3c")
    _ax.barh(_y, _pi, left=_pr + _pm, label="irregular", color="#f39c12")
    _ax.set_yticks(_y)
    _ax.set_yticklabels(_plot_df["region"])
    _ax.set_xlabel("% of rotation-eligible pixels in region")
    _ax.set_title("Rotation class mix by state (strict YAML, same labels as maps)")
    _ax.legend(loc="lower right", fontsize=9)
    _ax.set_xlim(0, 100)
    _fig.tight_layout()
    _fig_dir = REPO_ROOT / cfg["output"]["figures_dir"]
    _fig_dir.mkdir(parents=True, exist_ok=True)
    _pfig = _fig_dir / "task2__per_state_rotation_classes.png"
    _fig.savefig(_pfig, dpi=200, bbox_inches="tight")
    plt.show()
    print("Wrote", _pfig.relative_to(REPO_ROOT))

run_id = uuid.uuid4().hex[:8]
run_dir = REPO_ROOT / "artifacts" / "logs" / "runs" / run_id
run_dir.mkdir(parents=True, exist_ok=True)
bundle = {
    "task": "task2_crop_rotation",
    "run_id": run_id,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "config_path": "configs/task2_crop_rotation.yaml",
    "outputs": {
        "rotation_metrics": str((REPO_ROOT / cfg["output"]["processed_dir"] / "rotation_metrics.parquet").relative_to(REPO_ROOT)).replace("\\", "/"),
        "rotation_map_smoothed": str(sm_tif.relative_to(REPO_ROOT)).replace("\\", "/"),
        "areal_stats_csv": str(csv_path.relative_to(REPO_ROOT)).replace("\\", "/"),
        "areal_stats_metadata": str(meta_path.relative_to(REPO_ROOT)).replace("\\", "/"),
    },
}
if reg_csv is not None:
    bundle["outputs"]["areal_stats_by_region_csv"] = str(reg_csv.relative_to(REPO_ROOT)).replace("\\", "/")
(run_dir / "run_bundle.json").write_text(json.dumps(bundle, indent=2), encoding="utf-8")
print("Wrote", (run_dir / "run_bundle.json").relative_to(REPO_ROOT))


     rotation_class  pixel_count     area_ha  pct_of_valid
0  regular_rotation       570202  17669192.4         27.36
1       monoculture        81308   2519539.9          3.90
2         irregular      1432602  44392900.1         68.74
Wrote artifacts\tables\task4\task2__areal_stats_by_class__20260411.csv
Wrote artifacts\tables\task4\task2__areal_stats_by_class__20260411__metadata.json


Corn Belt boundaries unavailable (geopandas + Natural Earth or data/external/states/*.shp).
Single-region fallback: full_raster_extent (no per-state breakdown).


               region  n_pixels  pct_regular  pct_monoculture  pct_irregular
0  full_raster_extent   2084112        28.15             6.04          65.81
Wrote artifacts\tables\task4\task2__areal_stats_by_region__20260411.csv
Wrote artifacts\logs\runs\90458a3a\run_bundle.json
